# Veyra One — M1 Smoke Test (Kaggle)

**Zweck:** Pipeline-Validierung (Import, Forward, Backward, Checkpoint) auf Dummy-Daten.
Kein echtes Training — nur Code-Smoke vor dem Cluster-Lauf.

GPU erforderlich: *Accelerator → GPU T4/P100* + *Internet → On*.


In [ ]:
# 1. Environment check
import torch, sys, os
print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1),'GB')
assert torch.cuda.is_available(), 'Keine GPU! Accelerator auf GPU setzen.'

In [ ]:
# 2. Repo beziehen (anpassen!)
REPO = '/kaggle/working/veyra'
REPO_URL = 'https://github.com/Lianomeister/Veyra.git'
import os, shutil
if os.path.exists(REPO):
    shutil.rmtree(REPO)
!git clone --depth 1 $REPO_URL $REPO
%cd $REPO
print('CWD:', os.getcwd())

In [ ]:
# 3. Dependencies
!pip install -q -r requirements.txt
import yaml, omegaconf, transformers
print('deps ok')

In [ ]:
# 4. Dummy-Korpus
import os
os.makedirs('data/smoke', exist_ok=True)
text = ('The quick brown fox jumps over the lazy dog. ' * 5000)
open('data/smoke/smoke.txt','w').write(text)
print('corpus chars:', len(text))

In [ ]:
# 5. Tokenizer Smoke (1 Epoch, winzig)
!python training/train_tokenizer.py --config configs/eval/lm1b-small.yaml --data data/smoke/smoke.txt --out artifacts/smoke/tokenizer.pt --epochs 1 --batch 4 --max-chunks 20 --lr 1e-3

In [ ]:
# 6. Checkpoint prüfen (Tokenizer)
import torch
sd = torch.load('artifacts/smoke/tokenizer.pt', map_location='cpu')
print('tokenizer keys:', list(sd.keys()))
print('num params:', sum(v.numel() for v in sd['model'].values()))

In [ ]:
# 7. Generator Smoke (1 Epoch, winzig)
!python training/train_generator.py --config configs/eval/lm1b-small.yaml --tokenizer artifacts/smoke/tokenizer.pt --data data/smoke/smoke.txt --out artifacts/smoke/generator.pt --epochs 1 --batch 2 --max-chunks 20 --lr 1e-3

In [ ]:
# 8. Checkpoint prüfen (Generator)
import torch
sd = torch.load('artifacts/smoke/generator.pt', map_location='cpu')
print('generator keys:', list(sd.keys()))
print('num params:', sum(v.numel() for v in sd['model'].values()))

In [ ]:
# 9. Abschluss
print('Smoke test finished. Check outputs above for errors.')